In [1]:
import pandas as pd
import os
import gc
import logging
import time
from sqlalchemy import create_engine

from google.colab import drive
drive.mount('/content/drive')

BASE_PROJECT = "/content/drive/MyDrive/inventory_project"
DATA_PATH = os.path.join(BASE_PROJECT, "data")
LOG_PATH = os.path.join(BASE_PROJECT, "logs", "ingestion_db.log")
DB_PATH = os.path.join(BASE_PROJECT, "inventory.db")

os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)

# Remove any existing handlers added by Colab
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Create logger
logger = logging.getLogger("ingestion")
logger.setLevel(logging.INFO)

# Create handlers
file_handler = logging.FileHandler(LOG_PATH)
stream_handler = logging.StreamHandler()

# Create formatter
formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s"
)

# Attach formatter
file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)

# Attach handlers to logger
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

logger.info("--------------------Logging initialized--------------------")

engine = create_engine(f"sqlite:///{DB_PATH}")

def ingest_db(df, table_name):
    df.to_sql(
        table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=50_000
    )

def load_raw_data():
  for file in os.listdir(DATA_PATH):
    if not file.endswith(".csv"):
        continue

    file_path = os.path.join(DATA_PATH, file)
    table_name = file[:-4]

    start_time = time.time()
    logger.info(f"Ingesting file: {file} in db")

    try:
        # Read CSV
        df = pd.read_csv(file_path)

        # Normalize column names
        df.columns = (
            df.columns
            .str.lower()
            .str.strip()
            .str.replace(" ", "_")
        )

        logger.info(
            f"Loaded {file} | rows={df.shape[0]} | cols={df.shape[1]}"
        )

        # Load into database
        ingest_db(df, table_name)

        elapsed = (time.time() - start_time) / 60
        logger.info(
            f"Finished {file} -> table={table_name} "
            f"in {elapsed:.2f} minutes"
        )

        # Free memory
        del df
        gc.collect()

    except Exception:
        logger.exception(f"Error while processing file: {file}")

  logger.info("--------------------Ingestion Completed--------------------")

if __name__ == "__main__":
    load_raw_data()

Mounted at /content/drive


2026-05-12 07:39:55,821 | INFO | --------------------Logging initialized--------------------
2026-05-12 07:39:56,200 | INFO | Ingesting file: vendor_invoice.csv in db
2026-05-12 07:39:56,438 | INFO | Loaded vendor_invoice.csv | rows=5543 | cols=10
2026-05-12 07:40:29,172 | INFO | Finished vendor_invoice.csv -> table=vendor_invoice in 0.55 minutes
2026-05-12 07:40:29,324 | INFO | Ingesting file: purchases.csv in db
2026-05-12 07:40:43,514 | INFO | Loaded purchases.csv | rows=2372474 | cols=16
2026-05-12 07:42:40,065 | INFO | Finished purchases.csv -> table=purchases in 2.18 minutes
2026-05-12 07:42:40,319 | INFO | Ingesting file: purchase_prices.csv in db
2026-05-12 07:42:40,823 | INFO | Loaded purchase_prices.csv | rows=12261 | cols=9
2026-05-12 07:42:41,172 | INFO | Finished purchase_prices.csv -> table=purchase_prices in 0.01 minutes
2026-05-12 07:42:41,247 | INFO | Ingesting file: end_inventory.csv in db
2026-05-12 07:42:42,159 | INFO | Loaded end_inventory.csv | rows=224489 | cols=

In [2]:
# import pandas as pd
# import os
# import gc
# import logging
# import time
# from sqlalchemy import create_engine

# from google.colab import drive
# drive.mount('/content/drive')

# BASE_PROJECT = "/content/drive/MyDrive/inventory_project"
# DATA_PATH = os.path.join(BASE_PROJECT, "data")
# LOG_PATH = os.path.join(BASE_PROJECT, "logs", "ingestion_db.log")
# DB_PATH = os.path.join(BASE_PROJECT, "inventory.db")

# os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)

# # Remove any existing handlers added by Colab
# for handler in logging.root.handlers[:]:
#     logging.root.removeHandler(handler)

# # Create logger
# logger = logging.getLogger("ingestion")
# logger.setLevel(logging.INFO)

# # Create handlers
# file_handler = logging.FileHandler(LOG_PATH)
# stream_handler = logging.StreamHandler()

# # Create formatter
# formatter = logging.Formatter(
#     "%(asctime)s | %(levelname)s | %(message)s"
# )

# # Attach formatter
# file_handler.setFormatter(formatter)
# stream_handler.setFormatter(formatter)

# # Attach handlers to logger
# logger.addHandler(file_handler)
# logger.addHandler(stream_handler)

# logger.info("--------------------Logging initialized--------------------")

# engine = create_engine(f"sqlite:///{DB_PATH}")

# def ingest_db(df, table_name):
#     df.to_sql(
#         table_name,
#         con=engine,
#         if_exists="replace",
#         index=False,
#         chunksize=50_000
#     )

# def load_raw_data():
#   for file in os.listdir(DATA_PATH):
#     if not file.endswith(".csv"):
#         continue

#     file_path = os.path.join(DATA_PATH, file)
#     table_name = file[:-4]

#     start_time = time.time()
#     logger.info(f"Ingesting file: {file} in db")

#     try:
#         # Read CSV
#         df = pd.read_csv(file_path)

#         # Normalize column names
#         df.columns = (
#             df.columns
#             .str.lower()
#             .str.strip()
#             .str.replace(" ", "_")
#         )

#         logger.info(
#             f"Loaded {file} | rows={df.shape[0]} | cols={df.shape[1]}"
#         )

#         # Load into database
#         ingest_db(df, table_name)

#         elapsed = (time.time() - start_time) / 60
#         logger.info(
#             f"Finished {file} -> table={table_name} "
#             f"in {elapsed:.2f} minutes"
#         )

#         # Free memory
#         del df
#         gc.collect()

#     except Exception:
#         logger.exception(f"Error while processing file: {file}")

#   logger.info("--------------------Ingestion Completed--------------------")

# if __name__ == "__main__":
#     load_raw_data()

In [3]:
# !ls /content/drive/MyDrive/inventory_project/logs

In [4]:
# import pandas as pd
# import os
# import gc
# import logging
# import time
# from sqlalchemy import create_engine

# from google.colab import drive
# drive.mount('/content/drive')

# BASE_PROJECT = "/content/drive/MyDrive/inventory_project"
# DATA_PATH = os.path.join(BASE_PROJECT, "data")
# LOG_PATH = os.path.join(BASE_PROJECT, "logs", "ingestion_db.log")
# DB_PATH = os.path.join(BASE_PROJECT, "inventory.db")

# os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)

# logger = logging.getLogger("ingestion")
# logger.setLevel(logging.INFO)

# if not logger.handlers:
#     file_handler = logging.FileHandler(LOG_PATH)
#     stream_handler = logging.StreamHandler()

#     formatter = logging.Formatter(
#         "%(asctime)s | %(levelname)s | %(message)s"
#     )

#     file_handler.setFormatter(formatter)
#     stream_handler.setFormatter(formatter)

#     logger.addHandler(file_handler)
#     logger.addHandler(stream_handler)

# engine = create_engine(f"sqlite:///{DB_PATH}")

# def ingest_db(df, table_name):
#     df.to_sql(
#         table_name,
#         con=engine,
#         if_exists="replace",
#         index=False,
#         chunksize=50_000
#     )

# def load_raw_data():
#     for file in os.listdir(DATA_PATH):
#         if not file.endswith(".csv"):
#             continue

#         file_path = os.path.join(DATA_PATH, file)
#         table_name = file[:-4]

#         start_time = time.time()
#         logger.info(f"Starting file: {file}")

#         try:
#             df = pd.read_csv(file_path)

#             df.columns = (
#                 df.columns
#                 .str.lower()
#                 .str.strip()
#                 .str.replace(" ", "_")
#             )

#             logger.info(
#                 f"Loaded {file} | rows={df.shape[0]} | cols={df.shape[1]}"
#             )

#             ingest_db(df, table_name)

#             elapsed = (time.time() - start_time) / 60
#             logger.info(
#                 f"Finished {file} -> table={table_name} "
#                 f"in {elapsed:.2f} minutes"
#             )

#             del df
#             gc.collect()

#         except Exception:
#             logger.exception(f"Error while processing file: {file}")
